# mixLSTM Test Notebook

This notebook demonstrates how to create a small sample dataset, initialize `mixLSTM`, and run a forward pass.

## 1. Environment Setup

In [1]:
import random
import numpy as np
import torch

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.datasets.splitter import split_by_sample

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

Running on device: cpu


## 2. Create Sample Dataset

We create synthetic time-series samples with shape `(T, input_dim)` stored under the input key `series`.

In [2]:
# Dataset parameters
num_samples = 200
T = 50  # sequence length
input_dim = 3
n_classes = 5

samples = [
    {
        "patient_id": f"patient-{i}",
        "visit_id": "visit-0",
        "series": torch.randn(T, input_dim).numpy().tolist(),
        "label": int(i % n_classes),
    }
    for i in range(num_samples)
]

input_schema = {"series": "tensor"}
output_schema = {"label": "multiclass"}

dataset = create_sample_dataset(
    samples=samples,
    input_schema=input_schema,
    output_schema=output_schema,
    dataset_name="mixlstm_demo",
)

print(f"Created dataset with {len(dataset)} samples")
print(f"Input schema: {dataset.input_schema}")
print(f"Output schema: {dataset.output_schema}")

Label label vocab: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}
Created dataset with 200 samples
Input schema: {'series': 'tensor'}
Output schema: {'label': 'multiclass'}


## 3. Split Dataset

In [3]:
train_data, val_data, test_data = split_by_sample(dataset, [0.7, 0.15, 0.15], seed=SEED)

print(f"Train: {len(train_data)} samples")
print(f"Val: {len(val_data)} samples")
print(f"Test: {len(test_data)} samples")

train_loader = get_dataloader(train_data, batch_size=8, shuffle=True)
val_loader = get_dataloader(val_data, batch_size=8, shuffle=False)
test_loader = get_dataloader(test_data, batch_size=8, shuffle=False)

Train: 140 samples
Val: 30 samples
Test: 30 samples


## 4. Initialize `mixLSTM` Model

In [15]:
from pyhealth.models import MixLSTM

model = MixLSTM(dataset=dataset, num_experts=10, hidden_size=64)
model = model.to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

Model created with 180390 parameters


## 5. Test Forward Pass

In [ ]:
# Fetch a batch and run a forward pass
batch = next(iter(train_loader))

with torch.no_grad():
    outputs = model(**batch)

print("Output keys:", outputs.keys())
print(f"Loss: {outputs['loss'].item():.4f}")
print(f"Logits shape: {outputs['logit'].shape}")

Output keys: dict_keys(['logit', 'y_prob', 'loss', 'y_true'])
Loss: 1.1942
Logits shape: torch.Size([8, 5])
Predictions: tensor([[-1.6454, -1.5354, -1.8028, -1.0847, -2.4207],
        [-2.9820, -3.0563, -2.2066, -1.8453, -0.4554],
        [-1.8053, -2.3038, -2.2325, -0.6283, -2.3546],
        [-1.4553, -1.7886, -1.6593, -1.3155, -1.9599],
        [-1.1825, -1.4733, -2.4646, -2.1457, -1.3383],
        [-1.3650, -1.9290, -1.5502, -1.5415, -1.7541],
        [-1.6411, -1.6955, -1.3702, -2.0516, -1.4266],
        [-1.5647, -2.0988, -1.5074, -1.5615, -1.4399]])


## 6. Optional Training (Example)

The following is an example sketch for training using PyHealth's `Trainer`. Uncomment to run training.

In [16]:
from pyhealth.trainer import Trainer
trainer = Trainer(model=model, device=device)
trainer.train(
     train_dataloader=train_loader,
     val_dataloader=val_loader,
     optimizer_params={"lr": 1e-1},
     epochs=10,
     monitor="accuracy",
)
#print("Trainer example provided (commented).")

MixLSTM(
  (model): ExampleMowLSTM(
    (activation): LogSoftmax(dim=-1)
    (cells): ModuleList(
      (0-49): 50 x MoW(
        (experts): mowLSTM(
          (dropouts): ModuleList(
            (0): Dropout(p=0, inplace=False)
          )
          (h2o): MoO(
            (experts): ModuleList(
              (0-9): 10 x Linear(in_features=64, out_features=5, bias=True)
            )
            (gate): IDGate()
          )
          (activation): LogSoftmax(dim=-1)
          (rnns): ModuleList(
            (0): mowLSTM_(
              (input_weights): MoO(
                (experts): ModuleList(
                  (0-9): 10 x Linear(in_features=3, out_features=256, bias=True)
                )
                (gate): IDGate()
              )
              (hidden_weights): MoO(
                (experts): ModuleList(
                  (0-9): 10 x Linear(in_features=64, out_features=256, bias=True)
                )
                (gate): IDGate()
              )
            )
         

Epoch 0 / 10:   0%|          | 0/18 [00:00<?, ?it/s]

--- Train epoch-0, step-18 ---
loss: 1.8305


Evaluation: 100%|██████████| 4/4 [00:00<00:00, 17.43it/s]

--- Eval epoch-0, step-18 ---
accuracy: 0.1000
f1_macro: 0.0583
f1_micro: 0.1000
loss: 1.8131
New best accuracy score (0.1000) at epoch-0, step-18


...

--- Eval epoch-8, step-162 ---
accuracy: 0.2333
f1_macro: 0.2024
f1_micro: 0.2333
loss: 2.1482



Epoch 9 / 10:   0%|          | 0/18 [00:00<?, ?it/s]

--- Train epoch-9, step-180 ---
loss: 1.2292


Evaluation: 100%|██████████| 4/4 [00:00<00:00, 18.26it/s]

--- Eval epoch-9, step-180 ---
accuracy: 0.1667
f1_macro: 0.1484
f1_micro: 0.1667
loss: 2.0957
Loaded best model


## 7. Notes

- Adjust `input_dim`, `T`, and model hyperparameters to match your real dataset.
- `mixLSTM` expects input tensors of shape `(batch, seq_len, input_dim)`.
- If you encounter device mismatches, ensure tensors and model are on the same `device`.